# 9. EmbeddingsAus *Transformer von Grund auf*.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle# die zwei oder drei Dateien, die das Kapitel braucht. Wer das# Repository geklont hat, findet sie schon vor, und es wird# nichts heruntergeladen.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"PLACES = [".", "code", "../code"]for name in ["ch08_bpe.py", "ch09_embeddings.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Embeddings: die Tabelle, und was ihre Zahlen bedeuten.Hier wird zweierlei gezeigt. Erstens, dass eine Zeile in einer Tabelle nachzuschlagen dieselbe Operation ist, wie einen One-Hot-Vektor mit der Tabelle zu multiplizieren, und das macht die Tabelle trainierbar. Zweitens, dass Vektoren, die aus dem Kontext gelernt werden, nach dem Kontext geordnet sind, und das ist der ganze Grund, sie zu lernen.

In [ ]:
import numpy as npfrom ch08_bpe import CORPUSnp.set_printoptions(precision=3, suppress=True, floatmode="fixed")

In [ ]:
def build_vocabulary(text):    return sorted(set(text.split()))

In [ ]:
def one_hot(index, size):    vector = np.zeros(size)    vector[index] = 1.0    return vector

Zählen, wie oft jedes Wortpaar nah beieinander vorkommt.

In [ ]:
def cooccurrence(text, vocab, window=2):    index = {word: i for i, word in enumerate(vocab)}    counts = np.zeros((len(vocab), len(vocab)))    words = text.split()    for position, word in enumerate(words):        lo = max(0, position - window)        hi = min(len(words), position + window + 1)        for other in range(lo, hi):            if other != position:                counts[index[word], index[words[other]]] += 1.0    return counts

Die Zähltabelle auf wenige Zahlen pro Wort verdichten.

In [ ]:
def embed_by_svd(counts, dimensions=2):    smoothed = np.log1p(counts)    centred = smoothed - smoothed.mean(axis=0, keepdims=True)    u, s, _ = np.linalg.svd(centred, full_matrices=False)    return u[:, :dimensions] * s[:dimensions]

In [ ]:
def cosine(a, b):    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

In [ ]:
def nearest(vocab, vectors, word, count=3):    i = vocab.index(word)    scored = [(cosine(vectors[i], vectors[j]), vocab[j])              for j in range(len(vocab)) if j != i]    return sorted(scored, reverse=True)[:count]

In [ ]:
def main():    vocab = build_vocabulary(CORPUS)    size = len(vocab)    rng = np.random.default_rng(0)    table = np.round(rng.standard_normal((size, 4)), 2)    print("NACHSCHLAGEN IST EINE MATRIXMULTIPLIKATION")    index = vocab.index("katze")    print(f"  Vokabular aus {size} Wörtern, Embeddings aus 4 Zahlen")    print(f"  'katze' hat den Index {index}")    print("  nachgeschlagen  ", table[index])    print("  One-Hot mal Tabelle")    print("                  ", one_hot(index, size) @ table)    print("  identisch:", np.allclose(table[index], one_hot(index, size)                                      @ table))    print("  die zweite Form ist differenzierbar, und deshalb lässt sich")    print("  die Tabelle überhaupt lernen")    print("\nVEKTOREN, NUR AUS DEM KONTEXT GELERNT")    counts = cooccurrence(CORPUS, vocab)    vectors = embed_by_svd(counts, dimensions=2)    print("  Wort        je zwei Zahlen")    for word, vector in zip(vocab, vectors):        print(f"  {word:<11} {vector}")    print("\nNÄCHSTE NACHBARN NACH KOSINUS-ÄHNLICHKEIT")    for word in ["katze", "milch", "die"]:        close = nearest(vocab, vectors, word)        pairs = ", ".join(f"{w} {s:.2f}" for s, w in close)        print(f"  {word:<11} {pairs}")    print("\nDEM KOSINUS IST DIE LÄNGE EGAL")    a = np.array([1.0, 2.0])    print(f"  Kosinus von [1, 2] mit [2, 4]       {cosine(a, 2 * a):.3f}")    product = float(a @ (2 * a))    print(f"  Skalarprodukt derselben beiden      {product:.3f}")    print("  gleiche Richtung, verschiedene Längen, also ist der Kosinus")    print("  eins und das Skalarprodukt nicht")

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.